In [1]:
import pandas as pd
import litellm
import time
import os
import sys
import re

# --- 1. CONFIGURATION ---
MODELS = {"Gemini-Pro": {"model_name": "gemini/gemini-2.5-pro"}}
BENCHMARK_FILE = "100_HU_Riddles_Benchmark_Questions.tsv"
OUTPUT_FILE = "gemini_final_clean_100.csv"

SAFETY_CONFIG = [
    {"category": "HARM_CATEGORY_HARASSMENT", "threshold": "BLOCK_NONE"},
    {"category": "HARM_CATEGORY_HATE_SPEECH", "threshold": "BLOCK_NONE"},
    {"category": "HARM_CATEGORY_SEXUALLY_EXPLICIT", "threshold": "BLOCK_NONE"},
    {"category": "HARM_CATEGORY_DANGEROUS_CONTENT", "threshold": "BLOCK_NONE"}
]

def clean_pipe_output(raw_output):
    """Surgically removes labels like [Válasz] and splits the string."""
    if not raw_output:
        return "ISMERETLEN", "Nincs válasz"
    
    # Remove common labels even if the model ignores the instruction
    clean = re.sub(r'\[.*?Válasz.*?\]', '', raw_output, flags=re.IGNORECASE)
    clean = re.sub(r'\[.*?Indoklás.*?\]', '', clean, flags=re.IGNORECASE)
    clean = clean.replace('[', '').replace(']', '').replace('*', '').strip()

    if '|' in clean:
        parts = clean.split('|', 1)
        ans = parts[0].strip()
        reas = parts[1].strip()
        # Fallback if the model echoes '1-4 szó'
        if "1-4 szó" in ans or len(ans) < 1:
            return "HIBA", reas
        return ans, reas
    
    return clean[:40], "Nincs indoklás"

def run_gemini_zero_drift(topic, riddle):
    system_prompt = "Ön egy precíz magyar szakértő asszisztens."
    
    # PASS 1
    initial_query = f"Téma: {topic}\nFeladvány: {riddle}\n\nMi a megfejtés?"

    try:
        res1 = litellm.completion(
            model=MODELS["Gemini-Pro"]["model_name"],
            messages=[{"role": "system", "content": system_prompt},
                      {"role": "user", "content": initial_query}],
            temperature=0.0,
            safety_settings=SAFETY_CONFIG
        )
        initial_answer = res1.choices[0].message.content or "ISMERETLEN"

        # PASS 2: Explicitly removing brackets from instructions to stop 'drifting'
        verification_query = (
            f"Feladvány: {riddle}\n"
            f"Kezdeti tipp: {initial_answer}\n\n"
            "FELADAT: Ellenőrizd a tippet. Ha a tipp helyes, ismételd meg a választ. "
            "Ha hibás, javítsd ki a helyes magyar megoldásra.\n\n"
            "SZIGORÚ FORMÁTUM (Példa): \n"
            "Egri Bikavér | Ez egy híres vörösbor Eger környékéről.\n\n"
            "TILALOM: Ne használj szögletes zárójeleket. Ne írd, hogy 'Helyes' vagy 'A tipp jó'. "
            "Csak a válasz, egy függőleges vonal (|), majd az indoklás következzen."
        )

        res2 = litellm.completion(
            model=MODELS["Gemini-Pro"]["model_name"],
            messages=[{"role": "system", "content": system_prompt},
                      {"role": "user", "content": verification_query}],
            temperature=0.0,
            safety_settings=SAFETY_CONFIG
        )
        
        return res2.choices[0].message.content.strip()
        
    except Exception as e:
        return f"ERROR | {str(e)[:50]}"

# --- 2. MAIN EXECUTION ---
if __name__ == "__main__":
    print(f"🚀 Launching ZERO DRIFT benchmark (Clean Responses Only)...")
    
    try:
        df = pd.read_csv(BENCHMARK_FILE, sep='\t')
    except Exception as e:
        print(f"🛑 Error: {e}")
        sys.exit(1)

    results = []
    
    for index, row in df.iterrows():
        print(f"Processing ID {row['ID']}...", end=" ", flush=True)
        raw_output = run_gemini_zero_drift(row['topic'], row['riddle_text'])
        ans, reas = clean_pipe_output(raw_output)

        results.append({
            'ID': row['ID'], 
            'Expected': row['reference_answer'], 
            'Gemini_Answer': ans,
            'Gemini_Reasoning': reas
        })
        print(f"Done: {ans}")
        time.sleep(1)

    results_df = pd.DataFrame(results)
    results_df.to_csv(OUTPUT_FILE, index=False, sep='\t')
    
    print("\n" + "="*60)
    print("🏆 FINAL CLEAN RESULTS")
    print("="*60)
    print(results_df[['ID', 'Expected', 'Gemini_Answer']].to_markdown(index=False))

🚀 Launching ZERO DRIFT benchmark (Clean Responses Only)...
Processing ID 1... Done: Unicum
Processing ID 2... Done: Tibor, Debrecen
Processing ID 3... Done: kürtőskalács
Processing ID 4... Done: Szaloncukor
Processing ID 5... Done: Egri Bikavér
Processing ID 6... Done: Makó
Processing ID 7... Done: puli
Processing ID 8... Done: Teqball
Processing ID 9... Done: Halászlé
Processing ID 10... Done: Tiszavirágzás
Processing ID 11... Done: Vuk
Processing ID 12... Done: Vau Lajos
Processing ID 13... Done: Süsü, a sárkány
Processing ID 14... Done: Pat és Mat
Processing ID 15... Done: Ropsz és Eddie
Processing ID 16... Done: Pom Pom
Processing ID 17... Done: A pálcikaember-család (egy gyerekrajzon)
Processing ID 18... Done: Zénó
Processing ID 19... Done: Gombóc Artúr
Processing ID 20... Done: Xantus
Processing ID 21... Done: Színlelt megfutamodás és lovasíjászat
Processing ID 22... Done: Lehel kürtjének mondája
Processing ID 23... Done: Toldi Miklós
Processing ID 24... Done: Vajk
Processing ID 